# import libraries

In [7]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.registry import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy   
import time


import sys
sys.path.append('../')
import helper_functions as hf

# generate recommendations

In [8]:
n = hf.get_iteration_number()

for i in range(3):
    print("Very Important: Please Confirm the Iteration Number is Iteration " + str(n))

Very Important: Please Confirm the Iteration Number is Iteration 5
Very Important: Please Confirm the Iteration Number is Iteration 5
Very Important: Please Confirm the Iteration Number is Iteration 5


In [9]:
time_start = time.time()


df_design, ax_client = hf.run_optimizer(current_iteration=n, drug = "IBP", bopt = 1, n_trials=3)

time_end = time.time()
time_duration = round((time_end - time_start)/60,2)

print("Time taken for optimization: " + str(time_duration) + " mins")
print("Time taken for optimization: " + str(time_duration * 60) + " seconds")

**************************************************************************************************************

Generating Bayesian Optimization trialsfor
Drug name:  Ibuprofen IBP  | Iteration:  5

**************************************************************************************************************


KeyboardInterrupt: 

# process results

In [10]:
ax_client = hf.load_design_optimizer(n)
ax_client.experiment.trials

{0: Trial(experiment_name='drug_surfactant', index=0, status=TrialStatus.COMPLETED, arm=Arm(name='0_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 48, 's2': 59, 's3': 49, 's4': 31, 's5': 96, 's6': 8, 's7': 10, 's8': 34, 'surfactant_conc': 85, 'drug_conc': 77})),
 1: Trial(experiment_name='drug_surfactant', index=1, status=TrialStatus.COMPLETED, arm=Arm(name='1_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 95, 's2': 21, 's3': 75, 's4': 63, 's5': 42, 's6': 71, 's7': 55, 's8': 99, 'surfactant_conc': 35, 'drug_conc': 2})),
 2: Trial(experiment_name='drug_surfactant', index=2, status=TrialStatus.COMPLETED, arm=Arm(name='2_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 52, 's2': 99, 's3': 4, 's4': 13, 's5': 22, 's6': 89, 's7': 28, 's8': 57, 'surfactant_conc': 18, 'drug_conc': 33})),
 3: Trial(experiment_name='drug_surfactant', index=3, status=TrialStatus.COMPLETED, arm=Arm(name='3_0', paramete

In [11]:
df_conc, df_vol = hf.design_to_conc_to_vol (n)

In [12]:
plate_well = input("Enter the plate well starting well (e.g., F1): ")
deepplate_well = input("Enter the deep plate well starting well (e.g., F1): ")


print("Please confirm the following information:")
print("Wellplate will start at: " + plate_well)
print("Deep plate will start at: " + deepplate_well)

print()
print("*******************************************************")
print("Continue if correct, or rerun this cell if incorrect.")
print("*******************************************************")

Please confirm the following information:
Wellplate will start at: C7
Deep plate will start at: E7

*******************************************************
Continue if correct, or rerun this cell if incorrect.
*******************************************************


In [13]:

hf.generate_protocol(df_vol=df_vol, iteration=n, plate_well=plate_well, deepplate_well=deepplate_well)

✅ Successfully wrote to: protocol/otflex_5.py


In [14]:
df_absorbance = hf.process_absorbance(iteration=n, threshold=0.1)
df_absorbance

,trial_index,success
0,0,1
1,1,1
2,2,0


In [15]:
results = hf.build_results(n, df_conc, df_absorbance)
results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc,initial_drug_conc_surfactant_conc_ratio,success,micelle_drug_conc,complexity
0,15,0,0,0,100,0,0,0,0,5.00,0.25,0.050000,1,0.025,1
1,16,0,0,0,100,0,0,0,0,2.85,0.25,0.087719,1,0.025,1
2,17,0,0,0,100,0,0,0,100,0.05,22.25,445.000000,0,0.000,2


In [16]:
norm_results = hf.normalize_data(results, 'normalize')

In [17]:
norm_results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc,initial_drug_conc_surfactant_conc_ratio,success,micelle_drug_conc,complexity
0,15,0,0,0,100,0,0,0,0,5.00,0.25,0.000500,1.0,0.01,0.125
1,16,0,0,0,100,0,0,0,0,2.85,0.25,0.000877,1.0,0.01,0.125
2,17,0,0,0,100,0,0,0,100,0.05,22.25,4.450000,0.0,0.00,0.250


# load the results to the optimizer

In [18]:
ax_client = hf.load_data_to_optimizer(iteration = n, norm_results = norm_results)
ax_client

[INFO 06-25 15:17:12] ax.service.ax_client: Completed trial 15 with data: {'micelle_drug_conc': (0.01, None), 'success': (1.0, None), 'initial_drug_conc_surfactant_conc_ratio': (0.0005, None), 'complexity': (0.125, None)}.
[INFO 06-25 15:17:12] ax.service.ax_client: Completed trial 16 with data: {'micelle_drug_conc': (0.01, None), 'success': (1.0, None), 'initial_drug_conc_surfactant_conc_ratio': (0.000877, None), 'complexity': (0.125, None)}.
[INFO 06-25 15:17:12] ax.service.ax_client: Completed trial 17 with data: {'micelle_drug_conc': (0.0, None), 'success': (0.0, None), 'initial_drug_conc_surfactant_conc_ratio': (4.45, None), 'complexity': (0.25, None)}.


AxClient(experiment=Experiment(drug_surfactant))